In [1]:
"""!sudo apt update
!sudo apt install -y cmake libomp-dev
!pip install --upgrade pip
!pip install git+https://github.com/carlosluis/stable-baselines3@fix_tests
!pip install gymnasium "numpy<2.0" pandas pandas-ta matplotlib "tensorflow<=2.16" scipy scikit-learn optuna"""

'!sudo apt update\n!sudo apt install -y cmake libomp-dev\n!pip install --upgrade pip\n!pip install git+https://github.com/carlosluis/stable-baselines3@fix_tests\n!pip install gymnasium "numpy<2.0" pandas pandas-ta matplotlib "tensorflow<=2.16" scipy scikit-learn optuna'

In [2]:
from utils.envs import load_dataset
import pandas as pd

In [3]:
df = load_dataset('utils/envs/stocks_data/PYPL.csv')

In [4]:
df['Close']

0       100.47
1       101.42
2       102.67
3       102.03
4       102.12
         ...  
1253     79.30
1254     77.25
1255     78.13
1256     79.25
1257     81.41
Name: Close, Length: 1258, dtype: float64

In [5]:
X = df['Close'].pct_change()
X.iloc[0] = 0
X

0       0.000000
1       0.009456
2       0.012325
3      -0.006234
4       0.000882
          ...   
1253    0.013807
1254   -0.025851
1255    0.011392
1256    0.014335
1257    0.027256
Name: Close, Length: 1258, dtype: float64

In [6]:
TP = 0.1      # Take Profit (10%)
SL = 0.05      # Stop Loss (5%)
W = 15    # Window size (look ahead W ticks)
MAX_OFFSET = len(df)  # Maximum offset (use the length of the data as the limit)

In [7]:
def calculate_B(i, X, TP, SL, W, MAX_OFFSET):
    t = 1
    for j in range(i + 1, min(i + W, MAX_OFFSET)):  # Loop within window size or MAX_OFFSET
        t *= (1 + X[j])
        if t - 1 >= TP or t - 1 <= -SL:
            return t - 1  # Return the profit/loss if the threshold is met
    return t - 1  # If the loop completes, return the final value of t - 1

# Step 3: Apply the function to calculate the 'B' indicator for each index
B = []
for idx, value in X.items():
    B.append(calculate_B(idx, X, TP, SL, W, MAX_OFFSET))

# Convert the result into a pandas Series (if needed)
B = pd.Series(B, index=X.index)

In [8]:
from sb3.combined_env import CombinedEnv

In [11]:
env = CombinedEnv(df=df, window_size=10, frame_bound=(100, 140), no_action_punishment=0)

In [12]:
env.signal_features[0]

array([-0.18523932, -0.06692957, -0.70875734, -0.5730549 ,  0.28149304,
       -0.46460277,  0.08898366, -0.5157795 , -0.24943943, -0.39481795,
        0.15199733], dtype=float32)

In [14]:
len(env.B)

50